### Memoization
1. What is Memoization?
- Core Idea: Memoization is a time-saving technique that remembers function results instead of recomputing them.

- How it Works: When a function runs with a new input, the input is stored as a key in a dictionary, and the calculated output is stored as the value. Subsequent calls with the same input retrieve the result instantly via dictionary lookup.

2. @lru_cache

- Python’s built-in `@lru_cache` decorator from the functools module handles all caching automatically in a single line above the function definition.

3. Tracking Cache Performance with cache_info()
You can inspect how effectively the cache is working by calling `.cache_info()` on the memoized function:

- `hits`: Number of calls served straight from memory.

- `misses`: Number of calls computed from scratch.

- `currsize`: Total number of entries currently stored in the cache.

- `maxsize`: The upper limit of items the cache will hold.

4. Best Practices & Rules for Using Memoization
- When to Use: Ideal for slow, computationally expensive functions that are called repeatedly with identical arguments. For fast or rarely-repeated functions, the cache lookup overhead outweighs the time saved.

- Pure Functions Only: The function must be pure (the same inputs always return the exact same output). Impure functions (e.g., reading current time or random numbers) will store and return stale data.

- Hashable Arguments Required: Arguments must be immutable types (e.g., numbers, strings, tuples, frozen sets). Mutable types like list or dict trigger a TypeError.

- Cache Size Limits: By default, @lru_cache caps at 128 entries to prevent memory leakage. When full, it drops the Least Recently Used (LRU) item. To allow an unbounded cache, set maxsize=None .

5. Memoization for classes
- Using `@lru_cache` on class methods in Python requires extra care. The biggest trap is memory leaks: decorating a standard instance method creates a strong reference to self inside the global cache dictionary, preventing garbage collection of the entire class instance.

- Here are the essential good practices for using @lru_cache inside Python classes.

    - Avoid Decorating Instance Methods Directly
When you decorate a standard instance method with @lru_cache, self is passed as the first argument. The cache holds a reference to self, which keeps the instance in memory forever—even if it goes out of scope everywhere else.

```python
from functools import lru_cache

class DataProcessor:
    @lru_cache(maxsize=128)
    def process(self, value: int):
        # 'self' is cached as part of the key!
        return value * 42
```
   - Use `@cached_property` for zero-argument methods
```python
from functools import cached_property

class UserReport:
    def __init__(self, raw_data):
        self.raw_data = raw_data

@cached_property
def summary(self):
    # Computed once per instance, cleaned up when instance is deleted
    return heavy_computation(self.raw_data)
```
   - Use @staticmethod or helper functions, if the method takes parameters, isolate the pure logic into a @staticmethod or a top-level function so self isn't passed into the cache.

```python
from functools import lru_cache

class PricingEngine:
    def calculate_tax(self, amount: float, region: str) -> float:
        return self._pure_tax_calc(amount, region)

    @staticmethod
    @lru_cache(maxsize=256)
    def _pure_tax_calc(amount: float, region: str) -> float:
        # No 'self' reference -> No memory leak!
        return amount * lookup_tax_rate(region)
```


In [ ]:
# Manual example of a caching decorator

from time import sleep


def cache(func):
    args_mapping = {}
    def wrapper(*args, **kwargs):
        key = args
        if key in args_mapping:
            print("checking cache")
            return args_mapping.get(key, "not found")
        else:
            print("calculating")
            result = func(
                *args, **kwargs
            )
            args_mapping[key] = result
            return result

    return wrapper


@cache  # -> cache(double(x))
def double(x: int) -> int:
    """
    Return the double of x
    :param x: int
    :return: int
    """
    sleep(1)
    return x * 2


print("first result", double(6))
print("second result", double(6))
print("third result", double(8))


In [ ]:
# use builtin python lru_cache

from functools import lru_cache
import time


# maxsize=128 stores up to 128 distinct function inputs/outputs in memory
@lru_cache(maxsize=128)
def slow_square(n):
    time.sleep(1)
    return n * n


# 1. First call with n = 4: Runs the function (takes ~1 second)
print(slow_square(4))
# Output:
# Calculating square for 4...
# 16

# 2. Second call with n = 4: Returned instantly from cache!
print(slow_square(4))
# Output:
# 16

# 3. Call with a new input n = 5: Runs the function again
print(slow_square(5))
# Output:
# Calculating square for 5...
# 25

print(slow_square.cache_info())
# CacheInfo(hits=1, misses=2, maxsize=128, currsize=2)

slow_square.cache_clear()  # Resets the cache